# Proyecto Final - Modelo NLP
### Clasificación de reseñas de Amazon en español

## Paso 1: Cargar el dataframe

In [1]:
import subprocess
subprocess.run(["pip", "install", "fsspec", "huggingface_hub", "datasets"], check=True)

CompletedProcess(args=['pip', 'install', 'fsspec', 'huggingface_hub', 'datasets'], returncode=0)

In [2]:
import pandas as pd

splits = {
    'train': 'data/train-00000-of-00001.parquet',
    'validation': 'data/validation-00000-of-00001.parquet',
    'test': 'data/test-00000-of-00001.parquet'
}

df = pd.read_parquet("hf://datasets/KRadim/edit_amazon_reviews_multi_es/" + splits["train"])

print("Shape del dataframe:", df.shape)
df.head()




c:\Users\danie\OneDrive\Documents\Data Sciencie\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape del dataframe: (199500, 9)


,id,stars,review_body,review_title,language,product_category,lenght_review_body,lenght_review_title,lenght_product_category
0,111516,1,No llegaron las hélices,Mal servicio,es,electronics,23,12,11
1,107084,4,Me encanta lo ligera y manejable que es. Plega...,Manejable y ligera,es,baby_product,420,18,12
2,199386,1,"De las dos baterias , hay una que no funciona",No comprare mas,es,home_improvement,45,15,16
3,69470,2,"la iluminación es genial, pero el adhesivo pos...",el adhesivo no pega bien,es,home,105,24,4
4,114836,4,La he comprado porque por estética y colores e...,Original,es,home,165,8,4


## Paso 2: Seleccionar variables y reemplazar etiquetas

In [3]:
# Variable independiente: review_title
# Variable dependiente: stars

X = df['review_title']
y = df['stars']

# Reemplazar los valores numericos por etiquetas
y = y.replace({
    1: 'Muy malo',
    2: 'Malo',
    3: 'Regular',
    4: 'Bueno',
    5: 'Excelente'
})

print("Distribución de clases:")
print(y.value_counts())

Distribución de clases:
stars
Bueno        39935
Excelente    39927
Malo         39901
Regular      39889
Muy malo     39848
Name: count, dtype: int64


## Paso 3: Preprocesamiento de textos

In [4]:
import re

def preprocesar_texto(texto):
    # Convertir a minusculas
    texto = texto.lower()
    # Eliminar caracteres especiales y numeros
    texto = re.sub(r'[^a-záéíóúüñ\s]', '', texto)
    # Eliminar espacios extra
    texto = texto.strip()
    return texto

X = X.apply(preprocesar_texto)

print("Ejemplos preprocesados:")
print(X.head())

Ejemplos preprocesados:
0                mal servicio
1          manejable y ligera
2             no comprare mas
3    el adhesivo no pega bien
4                    original
Name: review_title, dtype: str


## Paso 4: Dividir en train y test

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Tamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

Tamaño entrenamiento: (159600,)
Tamaño prueba: (39900,)


## Paso 5: Crear Pipeline (vectorización + modelo)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('vectorizador', TfidfVectorizer()),
    ('modelo', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

print("Pipeline entrenado correctamente")

Pipeline entrenado correctamente


## Paso 6: Validar el modelo

In [7]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)

print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

Reporte de clasificación:
              precision    recall  f1-score   support

       Bueno       0.42      0.43      0.42      7897
   Excelente       0.55      0.58      0.56      7993
        Malo       0.43      0.40      0.42      8042
    Muy malo       0.59      0.63      0.61      7971
     Regular       0.38      0.34      0.36      7997

    accuracy                           0.48     39900
   macro avg       0.47      0.48      0.47     39900
weighted avg       0.47      0.48      0.47     39900



## Paso 7: Guardar el pipeline como .pkl

In [8]:
import joblib

joblib.dump(pipeline, 'pipeline.pkl')

print("Pipeline guardado como pipeline.pkl")

Pipeline guardado como pipeline.pkl


## Prueba rápida del pipeline guardado

In [9]:
modelo_cargado = joblib.load('pipeline.pkl')

textos_prueba = [
    "excelente producto, me encantó",
    "muy malo, no lo recomiendo",
    "es regular, nada especial"
]

predicciones = modelo_cargado.predict(textos_prueba)

for texto, pred in zip(textos_prueba, predicciones):
    print(f"Texto: '{texto}' → Predicción: {pred}")

Texto: 'excelente producto, me encantó' → Predicción: Excelente
Texto: 'muy malo, no lo recomiendo' → Predicción: Muy malo
Texto: 'es regular, nada especial' → Predicción: Malo
